
# NSD EDA — fMRI Visual Reconstruction Dataset

This notebook performs reproducible exploratory data analysis (EDA) and QC for a local
Natural Scenes Dataset (NSD) collection used in an fMRI visual-reconstruction workflow.

It is designed for NSD/MindEye-style files such as:

- `sub01_nsdgeneral.hdf5`, `sub02_nsdgeneral.hdf5`, ...
- `sub01_annots.npy`, `sub02_annots.npy`, ...
- `sub01_coco73k_idx.npy`, when available
- optional stimulus/image directories

### EDA covered

1. Environment and configuration
2. NSD file inventory
3. HDF5 structure inspection
4. Dataset-shape and metadata inspection
5. Annotation and COCO-index inspection
6. Candidate fMRI-array discovery
7. Memory-safe fMRI QC after a verified HDF5 key is selected
8. Stimulus inventory and sample display
9. Semantic annotation inspection
10. Memory-bounded PCA EDA
11. Cross-subject QC
12. Missing-data/integrity checks
13. Export of tables, figures, and a final JSON summary

> **Scientific note:** this notebook does not invent ROI labels, GLM activations, stimulus timing,
> or HDF5 dataset names. It discovers the local structure and requires the user to verify the
> fMRI dataset key before numerical fMRI analysis.


In [ ]:

# ============================================================
# 1. CONFIGURATION
# ============================================================

from pathlib import Path
import os
import sys
import json
import math
import random
import warnings
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Change this to the directory containing your NSD files.
# Example:
# NSD_ROOT = Path(r"D:/fMRI-visual-reconstruction-GitHub/data/nsd")
NSD_ROOT = Path(".")

# Optional stimulus directory. Set to a Path or string when available.
# Example:
# STIMULUS_ROOT = NSD_ROOT / "stimuli"
STIMULUS_ROOT = None

EDA_ROOT = NSD_ROOT / "eda_outputs"
FIG_DIR = EDA_ROOT / "figures"
TABLE_DIR = EDA_ROOT / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Adjust this list to the subjects actually present.
SUBJECTS = [1, 2, 3, 4, 5, 6, 7, 8]

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Python:", sys.version)
print("NSD root:", NSD_ROOT.resolve())
print("EDA output:", EDA_ROOT.resolve())


In [ ]:

# ============================================================
# 2. PACKAGE CHECK
# ============================================================

packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "h5py": "h5py",
    "PIL": "Pillow",
    "sklearn": "scikit-learn",
}

missing = []

for module_name, package_name in packages.items():
    try:
        __import__(module_name)
    except Exception:
        missing.append(package_name)

if missing:
    print("Missing packages:", sorted(set(missing)))
    print("%pip install numpy pandas matplotlib h5py pillow scikit-learn")
else:
    print("All EDA packages are available.")



## 3. NSD file discovery

The notebook discovers files rather than assuming every subject exists. This is useful when
the repository contains only a subset of NSD subjects.


In [ ]:

# ============================================================
# 3. FILE INVENTORY
# ============================================================

def find_first(patterns):
    for pattern in patterns:
        matches = sorted(NSD_ROOT.glob(pattern))
        if matches:
            return matches[0]
    return None

records = []

for sid in SUBJECTS:
    h5 = find_first([
        f"sub{sid:02d}_nsdgeneral.hdf5",
        f"sub{sid:02d}_nsdgeneral.h5",
        f"sub{sid}_nsdgeneral.hdf5",
        f"sub{sid}_nsdgeneral.h5",
        f"**/sub{sid:02d}_nsdgeneral.hdf5",
        f"**/sub{sid:02d}_nsdgeneral.h5",
    ])

    annot = find_first([
        f"sub{sid:02d}_annots.npy",
        f"sub{sid}_annots.npy",
        f"**/sub{sid:02d}_annots.npy",
        f"**/sub{sid}_annots.npy",
    ])

    coco = find_first([
        f"sub{sid:02d}_coco73k_idx.npy",
        f"sub{sid}_coco73k_idx.npy",
        f"**/sub{sid:02d}_coco73k_idx.npy",
        f"**/sub{sid}_coco73k_idx.npy",
    ])

    records.append({
        "subject": sid,
        "hdf5": str(h5) if h5 else None,
        "annotations": str(annot) if annot else None,
        "coco73k_index": str(coco) if coco else None,
        "hdf5_exists": h5 is not None,
        "annotations_exists": annot is not None,
        "coco_index_exists": coco is not None,
    })

inventory = pd.DataFrame(records)
display(inventory)

inventory.to_csv(
    TABLE_DIR / "nsd_file_inventory.csv",
    index=False
)



## 4. HDF5 structure inspection

This section is read-only. It recursively lists groups and datasets, including shape, dtype and
size. This is intentionally done before selecting an fMRI array so that the notebook does not
silently assume an incorrect internal key.


In [ ]:

# ============================================================
# 4. HDF5 STRUCTURE
# ============================================================

import h5py

def describe_hdf5(path, max_items=1000):
    rows = []

    def visitor(name, obj):
        if len(rows) >= max_items:
            return

        if isinstance(obj, h5py.Dataset):
            rows.append({
                "path": name,
                "type": "dataset",
                "shape": tuple(obj.shape),
                "dtype": str(obj.dtype),
                "ndim": obj.ndim,
                "size": int(obj.size),
            })

        elif isinstance(obj, h5py.Group):
            rows.append({
                "path": name,
                "type": "group",
                "shape": None,
                "dtype": None,
                "ndim": None,
                "size": None,
            })

    with h5py.File(path, "r") as f:
        f.visititems(visitor)

    return pd.DataFrame(rows)

h5_tables = {}

for sid in SUBJECTS:
    row = inventory.loc[inventory["subject"] == sid].iloc[0]
    path = row["hdf5"]

    if not path:
        continue

    print("\n" + "=" * 90)
    print(f"SUBJECT {sid}: {path}")
    print("=" * 90)

    table = describe_hdf5(path)
    h5_tables[sid] = table

    display(table)

    table.to_csv(
        TABLE_DIR / f"sub{sid:02d}_hdf5_structure.csv",
        index=False
    )


In [ ]:

# ============================================================
# 5. COMBINED HDF5 DATASET SUMMARY
# ============================================================

parts = []

for sid, table in h5_tables.items():
    if not table.empty:
        t = table[table["type"] == "dataset"].copy()
        t.insert(0, "subject", sid)
        parts.append(t)

if parts:
    h5_summary = pd.concat(parts, ignore_index=True)
    display(h5_summary)
    h5_summary.to_csv(
        TABLE_DIR / "nsd_hdf5_dataset_summary.csv",
        index=False
    )
else:
    h5_summary = pd.DataFrame()
    print("No HDF5 datasets discovered.")



## 6. Annotation and COCO-index inspection

The notebook loads the local annotation arrays and reports their shape, dtype and representative
records. It does not assume a fixed annotation schema.


In [ ]:

# ============================================================
# 6. ANNOTATIONS / COCO INDICES
# ============================================================

annotation_objects = {}
coco_objects = {}

for sid in SUBJECTS:
    row = inventory.loc[inventory["subject"] == sid].iloc[0]

    annot_path = row["annotations"]
    coco_path = row["coco73k_index"]

    print("\n" + "=" * 90)
    print(f"SUBJECT {sid}")
    print("=" * 90)

    if annot_path:
        try:
            arr = np.load(annot_path, allow_pickle=True)
            annotation_objects[sid] = arr

            print("Annotations")
            print(" path :", annot_path)
            print(" shape:", getattr(arr, "shape", None))
            print(" dtype:", getattr(arr, "dtype", None))

            flat = np.asarray(arr).reshape(-1)
            print(" entries:", flat.size)
            print(" first entries:", flat[:10])
        except Exception as e:
            print("Annotation load error:", repr(e))
    else:
        print("No annotation file found.")

    if coco_path:
        try:
            arr = np.load(coco_path, allow_pickle=True)
            coco_objects[sid] = arr

            print("\nCOCO 73K index")
            print(" path :", coco_path)
            print(" shape:", getattr(arr, "shape", None))
            print(" dtype:", getattr(arr, "dtype", None))

            flat = np.asarray(arr).reshape(-1)
            print(" entries:", flat.size)
            print(" first entries:", flat[:10])
        except Exception as e:
            print("COCO index load error:", repr(e))
    else:
        print("No COCO 73K index found.")



## 7. Candidate numeric HDF5 arrays

The following cell identifies numeric datasets that could be relevant to fMRI features. It does
not select one automatically. Inspect the resulting table and set `FMRI_H5_KEY` below only after
confirming the correct dataset path for your NSD files.


In [ ]:

# ============================================================
# 7. CANDIDATE FMRI DATASETS
# ============================================================

def candidate_numeric_datasets(path):
    rows = []

    with h5py.File(path, "r") as f:
        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                if np.issubdtype(obj.dtype, np.number):
                    rows.append({
                        "path": name,
                        "shape": tuple(obj.shape),
                        "dtype": str(obj.dtype),
                        "ndim": obj.ndim,
                        "size": int(obj.size),
                    })
        f.visititems(visitor)

    return pd.DataFrame(rows)

candidate_tables = {}

for sid in SUBJECTS:
    path = inventory.loc[
        inventory["subject"] == sid, "hdf5"
    ].iloc[0]

    if not path:
        continue

    table = candidate_numeric_datasets(path)
    candidate_tables[sid] = table

    print(f"\nSubject {sid}")
    display(table)


In [ ]:

# ============================================================
# 8. VERIFY THE FMRI HDF5 KEY
# ============================================================

# IMPORTANT:
# Set this only after inspecting the candidate tables above.
#
# Example:
# FMRI_H5_KEY = "your/verified/path"
#
# Do not let the notebook guess this.

FMRI_H5_KEY = None

if FMRI_H5_KEY is None:
    print(
        "FMRI_H5_KEY is None. "
        "Large-array fMRI analysis is intentionally disabled."
    )
else:
    print("Verified fMRI HDF5 key:", FMRI_H5_KEY)



## 9. Memory-safe fMRI QC

This section uses chunked HDF5 reads. It never calls `get_fdata()` on the complete NSD array.
The first dimension is treated as the sample/trial dimension for the purpose of chunked statistics;
verify that interpretation from the HDF5 shape before using the output.


In [ ]:

# ============================================================
# 9. MEMORY-SAFE FMRI STATISTICS
# ============================================================

def h5_basic_statistics(path, key, chunk_rows=8):
    with h5py.File(path, "r") as f:
        dset = f[key]

        if not np.issubdtype(dset.dtype, np.number):
            raise TypeError(f"Dataset {key} is not numeric.")

        shape = tuple(dset.shape)
        dtype = str(dset.dtype)

        count = 0
        mean = 0.0
        m2 = 0.0
        min_value = np.inf
        max_value = -np.inf
        nan_count = 0
        inf_count = 0

        for start in range(0, shape[0], chunk_rows):
            end = min(start + chunk_rows, shape[0])

            block = np.asarray(
                dset[start:end],
                dtype=np.float32
            )

            nan_count += int(np.isnan(block).sum())
            inf_count += int(np.isinf(block).sum())

            values = block[np.isfinite(block)].astype(np.float64)

            if values.size:
                min_value = min(min_value, float(values.min()))
                max_value = max(max_value, float(values.max()))

                # Batch Welford update.
                n_b = values.size
                mean_b = float(values.mean())
                m2_b = float(((values - mean_b) ** 2).sum())

                if count == 0:
                    count = n_b
                    mean = mean_b
                    m2 = m2_b
                else:
                    delta = mean_b - mean
                    new_count = count + n_b
                    mean = mean + delta * n_b / new_count
                    m2 = m2 + m2_b + delta * delta * count * n_b / new_count
                    count = new_count

            del block
            del values

    std = np.sqrt(m2 / (count - 1)) if count > 1 else np.nan

    return {
        "shape": shape,
        "dtype": dtype,
        "finite_count": count,
        "nan_count": nan_count,
        "inf_count": inf_count,
        "min": min_value if count else np.nan,
        "max": max_value if count else np.nan,
        "mean": mean if count else np.nan,
        "std": std,
    }

if FMRI_H5_KEY is not None:
    rows = []

    for sid in SUBJECTS:
        path = inventory.loc[
            inventory["subject"] == sid, "hdf5"
        ].iloc[0]

        if not path:
            continue

        try:
            stats = h5_basic_statistics(
                path,
                FMRI_H5_KEY,
                chunk_rows=4
            )
            stats["subject"] = sid
            rows.append(stats)
        except Exception as e:
            print(
                f"Subject {sid}: fMRI statistics failed:",
                repr(e)
            )

    if rows:
        fmri_stats = pd.DataFrame(rows)
        fmri_stats = fmri_stats[
            [
                "subject", "shape", "dtype", "finite_count",
                "nan_count", "inf_count", "min", "max",
                "mean", "std"
            ]
        ]
        display(fmri_stats)
        fmri_stats.to_csv(
            TABLE_DIR / "nsd_fmri_statistics.csv",
            index=False
        )
else:
    print("Skipped because FMRI_H5_KEY is not verified.")



## 10. Stimulus inventory

Set `STIMULUS_ROOT` in the configuration cell if the local repository contains NSD stimulus
images. This section records dimensions and file metadata without loading all images at once.


In [ ]:

# ============================================================
# 10. STIMULUS INVENTORY
# ============================================================

from PIL import Image

if STIMULUS_ROOT is None:
    print("STIMULUS_ROOT is None. Stimulus inventory skipped.")
else:
    STIMULUS_ROOT = Path(STIMULUS_ROOT)

    extensions = {
        ".jpg", ".jpeg", ".png", ".bmp",
        ".webp", ".tif", ".tiff"
    }

    image_paths = [
        p for p in STIMULUS_ROOT.rglob("*")
        if p.is_file() and p.suffix.lower() in extensions
    ]

    rows = []

    for p in image_paths:
        try:
            with Image.open(p) as im:
                rows.append({
                    "path": str(p),
                    "filename": p.name,
                    "width": im.width,
                    "height": im.height,
                    "mode": im.mode,
                    "format": im.format,
                    "filesize_bytes": p.stat().st_size,
                })
        except Exception as e:
            rows.append({
                "path": str(p),
                "filename": p.name,
                "width": None,
                "height": None,
                "mode": None,
                "format": None,
                "filesize_bytes": p.stat().st_size,
                "error": repr(e),
            })

    stimulus_inventory = pd.DataFrame(rows)

    print("Stimulus files:", len(stimulus_inventory))
    display(stimulus_inventory.head(20))

    stimulus_inventory.to_csv(
        TABLE_DIR / "nsd_stimulus_inventory.csv",
        index=False
    )


In [ ]:

# ============================================================
# 11. RANDOM STIMULUS DISPLAY
# ============================================================

if STIMULUS_ROOT is None:
    print("Skipped: STIMULUS_ROOT is None.")
elif len(image_paths) == 0:
    print("No stimulus images found.")
else:
    n_show = min(12, len(image_paths))
    rng = np.random.default_rng(SEED)
    selected = list(
        rng.choice(
            image_paths,
            size=n_show,
            replace=False
        )
    )

    cols = 4
    rows = math.ceil(n_show / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(12, 3 * rows)
    )

    axes = np.atleast_1d(axes).ravel()

    for ax, path in zip(axes, selected):
        with Image.open(path) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(path.name, fontsize=8)
        ax.axis("off")

    for ax in axes[n_show:]:
        ax.axis("off")

    fig.suptitle(
        "Random NSD Stimulus Samples",
        fontsize=15,
        fontweight="bold"
    )

    plt.tight_layout()

    fig.savefig(
        FIG_DIR / "nsd_random_stimuli.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()



## 12. Semantic annotation inspection

This cell prints representative annotation records and, when the records are dictionary-like,
creates a DataFrame. It does not assume specific COCO/category fields unless they actually occur.


In [ ]:

# ============================================================
# 12. SEMANTIC ANNOTATION SUMMARY
# ============================================================

def summarize_annotations(obj, subject_id):
    arr = np.asarray(obj)
    flat = arr.reshape(-1)

    print(f"Subject {subject_id}")
    print("shape:", arr.shape)
    print("dtype:", arr.dtype)
    print("entries:", flat.size)

    for item in flat[:10]:
        print(repr(item))

    dict_records = [
        item for item in flat
        if isinstance(item, dict)
    ]

    if not dict_records:
        print(
            "No dictionary-like annotation records detected."
        )
        return pd.DataFrame()

    df = pd.DataFrame(dict_records)

    print("Detected fields:")
    print(list(df.columns))
    display(df.head())

    return df

semantic_tables = {}

for sid, obj in annotation_objects.items():
    try:
        df = summarize_annotations(
            obj,
            sid
        )
        if not df.empty:
            semantic_tables[sid] = df
            df.to_csv(
                TABLE_DIR / f"sub{sid:02d}_semantic_annotations.csv",
                index=False
            )
    except Exception as e:
        print(
            f"Subject {sid}: annotation summary failed:",
            repr(e)
        )



## 13. Memory-bounded fMRI PCA

PCA is an EDA tool here, not the shared-alignment model. It is run on a bounded random sample
after `FMRI_H5_KEY` is verified.


In [ ]:

# ============================================================
# 13. MEMORY-BOUNDED PCA
# ============================================================

if FMRI_H5_KEY is None:
    print("Skipped: FMRI_H5_KEY is not verified.")
else:
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler

    MAX_SAMPLES = 500
    MAX_FEATURES = 3000

    def sample_h5_rows(
        path,
        key,
        max_samples=500,
        max_features=3000
    ):
        with h5py.File(path, "r") as f:
            dset = f[key]

            n = dset.shape[0]
            n_take = min(n, max_samples)

            rng = np.random.default_rng(SEED)
            indices = np.sort(
                rng.choice(
                    n,
                    size=n_take,
                    replace=False
                )
            )

            x = np.asarray(
                dset[indices],
                dtype=np.float32
            )

        x = x.reshape(
            x.shape[0],
            -1
        )

        if x.shape[1] > max_features:
            idx = np.linspace(
                0,
                x.shape[1] - 1,
                max_features,
                dtype=int
            )
            x = x[:, idx]

        x = np.nan_to_num(
            x,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        return x

    pca_rows = []

    for sid in SUBJECTS:
        path = inventory.loc[
            inventory["subject"] == sid,
            "hdf5"
        ].iloc[0]

        if not path:
            continue

        try:
            x = sample_h5_rows(
                path,
                FMRI_H5_KEY,
                MAX_SAMPLES,
                MAX_FEATURES
            )

            if x.shape[0] < 3:
                continue

            x = StandardScaler().fit_transform(x)

            pca = PCA(
                n_components=2,
                random_state=SEED
            )

            z = pca.fit_transform(x)

            pca_rows.append({
                "subject": sid,
                "PC1_variance_ratio": pca.explained_variance_ratio_[0],
                "PC2_variance_ratio": pca.explained_variance_ratio_[1],
            })

            plt.figure(figsize=(6, 5))
            plt.scatter(
                z[:, 0],
                z[:, 1],
                s=12,
                alpha=0.65
            )
            plt.xlabel("PC1")
            plt.ylabel("PC2")
            plt.title(
                f"NSD Subject {sid} — fMRI PCA EDA"
            )
            plt.tight_layout()
            plt.savefig(
                FIG_DIR / f"sub{sid:02d}_fmri_pca.png",
                dpi=300,
                bbox_inches="tight"
            )
            plt.show()

            del x, z
            gc.collect()

        except Exception as e:
            print(
                f"Subject {sid}: PCA skipped:",
                repr(e)
            )

    if pca_rows:
        pca_summary = pd.DataFrame(pca_rows)
        display(pca_summary)
        pca_summary.to_csv(
            TABLE_DIR / "nsd_fmri_pca_summary.csv",
            index=False
        )



## 14. Cross-subject QC


In [ ]:

# ============================================================
# 14. CROSS-SUBJECT QC
# ============================================================

rows = []

for sid in SUBJECTS:
    row = inventory.loc[
        inventory["subject"] == sid
    ].iloc[0]

    item = {
        "subject": sid,
        "hdf5_exists": bool(row["hdf5_exists"]),
        "annotations_exists": bool(row["annotations_exists"]),
        "coco_index_exists": bool(row["coco_index_exists"]),
        "hdf5_datasets": (
            int((h5_tables[sid]["type"] == "dataset").sum())
            if sid in h5_tables else 0
        ),
        "annotation_entries": (
            int(np.asarray(annotation_objects[sid]).size)
            if sid in annotation_objects else np.nan
        ),
        "coco_index_entries": (
            int(np.asarray(coco_objects[sid]).size)
            if sid in coco_objects else np.nan
        ),
    }

    rows.append(item)

cross_subject_qc = pd.DataFrame(rows)
display(cross_subject_qc)

cross_subject_qc.to_csv(
    TABLE_DIR / "nsd_cross_subject_qc.csv",
    index=False
)



## 15. Missing-data and integrity report


In [ ]:

# ============================================================
# 15. INTEGRITY REPORT
# ============================================================

integrity_rows = []

for sid in SUBJECTS:
    row = inventory.loc[
        inventory["subject"] == sid
    ].iloc[0]

    issues = []

    if not row["hdf5_exists"]:
        issues.append("missing_hdf5")

    if not row["annotations_exists"]:
        issues.append("missing_annotations")

    if not row["coco_index_exists"]:
        issues.append("missing_coco73k_index")

    integrity_rows.append({
        "subject": sid,
        "status": "OK" if not issues else "CHECK",
        "issues": "; ".join(issues),
    })

integrity_df = pd.DataFrame(integrity_rows)
display(integrity_df)

integrity_df.to_csv(
    TABLE_DIR / "nsd_integrity_report.csv",
    index=False
)



## 16. Final EDA summary

All generated tables and figures are written below `eda_outputs/`. The original NSD files are
never modified.


In [ ]:

# ============================================================
# 16. FINAL SUMMARY
# ============================================================

summary = {
    "nsd_root": str(NSD_ROOT.resolve()),
    "subjects_requested": SUBJECTS,
    "subjects_with_hdf5": [
        int(x) for x in inventory.loc[
            inventory["hdf5_exists"], "subject"
        ].tolist()
    ],
    "subjects_with_annotations": [
        int(x) for x in inventory.loc[
            inventory["annotations_exists"], "subject"
        ].tolist()
    ],
    "subjects_with_coco_index": [
        int(x) for x in inventory.loc[
            inventory["coco_index_exists"], "subject"
        ].tolist()
    ],
    "verified_fmri_h5_key": FMRI_H5_KEY,
    "stimulus_root": str(STIMULUS_ROOT) if STIMULUS_ROOT else None,
}

with open(
    EDA_ROOT / "nsd_eda_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print(json.dumps(summary, indent=2))

print("\nGenerated locations:")
print("Figures:", FIG_DIR.resolve())
print("Tables :", TABLE_DIR.resolve())
print("Summary:", (EDA_ROOT / "nsd_eda_summary.json").resolve())



# Interpretation checklist

Before using NSD in the reconstruction pipeline:

- [ ] HDF5 files open successfully.
- [ ] HDF5 dataset paths have been inspected.
- [ ] The fMRI representation used for training is explicitly verified.
- [ ] Subject/sample dimensions are documented.
- [ ] Annotation ordering is checked against the intended samples.
- [ ] COCO indices are checked against the intended stimulus mapping.
- [ ] Stimulus files are readable when required.
- [ ] Missing subjects/trials/stimuli are documented.
- [ ] Brain masks/ROIs are used only when actually available.
- [ ] No raw-BOLD visualization is described as statistical activation.
- [ ] Train/test separation is preserved before retrieval and representation learning.
- [ ] The same preprocessing convention is applied across subjects.

This EDA notebook is the QC stage before the NSD preprocessing, shared latent alignment,
CLIP/DINOv2 projection, retrieval, and diffusion-reconstruction notebooks.
